# Build Your First Quant Trading Strategy
### QuantSoc workshop · 2 hours · [quant-soc.com](https://quant-soc.com)

By the end of this session you will have built a **complete, small systematic trading system**:

| Stage | What you do | Time |
|---|---|---|
| 1. Understand the market | load 5-minute price bars, compute returns | 8–18 min |
| 2. Find a candidate signal | build a *momentum* feature and test a hypothesis | 18–33 min |
| 3. Train a predictor | fit a linear model against a zero-forecast baseline | 33–50 min |
| 4. Build a portfolio | turn forecasts into weights with risk limits | 50–68 min |
| 5. Evaluate | backtest with costs, turnover and drawdown | 68–80 min |
| 6. Develop the application | refactor into `strategy.py`, export code + model | 80–105 min |
| 7. Connect and run | accelerated replay; optional supervised paper trading | 105–115 min |
| 8. Reveal and extend | how the market was generated; stress test | 115–120 min |

**Pipeline:** `bars → features → forecasts → target weights → orders → account`.
Every arrow is a function you will either write (six short exercises) or receive.

> **Important disclosure.** The five assets in this notebook (AURA, BOLT, CRUX, DUNE, ECHO) are **fictional**.
> Their prices come from a controlled *teaching* market generated by code, so that a small, learnable pattern
> definitely exists. Real markets do not come with that guarantee. Systematic trading is one part of quantitative
> finance (alongside pricing, risk, market-making and research); finishing this workshop gives you a working
> *system*, **not** a validated real-market investment strategy. The exact generating mechanism is revealed in the
> last section.

**Two ways to run this notebook**
* **Browser only (recommended):** Google Colab or Kaggle, standard CPU runtime. Everything runs inside this notebook.
* **Local:** clone the repository and run the same notebook in Jupyter; you can then launch the local runner and dashboard.

**How the exercises work.** Each exercise cell has a `# YOUR CODE` marker. Run the cell: the tracker `ex` checks
your answer and prints `[ok]` or a specific reason it is not right. `ex.hint(n)` reveals hints one at a time;
`ex.show_solution(n)` prints the reference code; `ex.use_reference(n)` lets you continue with the reference
solution, which is always labelled as such and never silently swapped in.

In [ ]:
# @title Setup (run once - safe to run again; it never overwrites your own files)
OWNER, REPO, REVISION = "z1nare", "quant_workshop_v1", "v1.0.0"   # pinned workshop release (see tools/set_release.py)

import io, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = not IN_COLAB and ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/input").is_dir())
if IN_COLAB or IN_KAGGLE:
    ROOT = Path("/content/workshop") if IN_COLAB else Path("/kaggle/working/workshop")
    marker = ROOT / ".release"
    wanted = f"{OWNER}/{REPO}@{REVISION}"
    if not (marker.exists() and marker.read_text().strip() == wanted):
        # Kaggle: prefer the workshop files attached to the notebook as a dataset (works with Internet off)
        attached = next((p for depth in range(1, 6) for p in Path("/kaggle/input").glob("*/" * depth + "quantsoc/__init__.py")),
                        None) if IN_KAGGLE else None
        tmp = ROOT.parent / "_workshop_src"
        if attached is not None:
            src = attached.parent.parent
            print("Using the workshop files attached to this notebook:", src)
        else:
            if OWNER.startswith("YOUR-"):
                raise SystemExit("The release settings are placeholders. Ask the instructor for the published notebook link.")
            url = f"https://github.com/{OWNER}/{REPO}/archive/{REVISION}.zip"
            print("Downloading workshop files from", url)
            try:
                payload = urllib.request.urlopen(url, timeout=60).read()
            except OSError as e:
                hint = (" On Kaggle: attach the workshop dataset (Add Input) or switch Internet on in the notebook settings,"
                        " then re-run this cell." if IN_KAGGLE else "")
                raise SystemExit(f"Could not download the workshop files ({e}).{hint}")
            with zipfile.ZipFile(io.BytesIO(payload)) as zf:
                shutil.rmtree(tmp, ignore_errors=True); zf.extractall(tmp)
            src = tmp / zf.namelist()[0].split("/")[0]
        ROOT.mkdir(exist_ok=True)
        # library, data and app files are refreshed; anything under workspace/ (your strategy, exports) is never touched
        for item in ["quantsoc", "data", "docs", "strategy.py", "run_trader.py", "dashboard.py",
                     "generate_data.py", "requirements.txt", "requirements-colab.txt", ".env.example"]:
            s, d = src / item, ROOT / item
            if s.is_dir():
                shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
            elif s.exists():
                shutil.copy(s, d)
        shutil.rmtree(tmp, ignore_errors=True)
        print("Installing the two extra packages the notebook image does not ship (alpaca-py, python-dotenv) ...")
        fail_fast = ["--retries", "0", "--timeout", "5"] if attached is not None else []   # Kaggle may have Internet off
        pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fail_fast, "-r", str(ROOT / "requirements-colab.txt")])
        if pip.returncode != 0:
            print("Could not install them (no internet?). Only the optional paper-trading demo needs them; everything else works.")
        marker.write_text(wanted)
        print("Done.")
else:
    here = Path.cwd().resolve()
    ROOT = next(p for p in [here, *here.parents] if (p / "quantsoc" / "__init__.py").exists())

WORK = ROOT / "workspace"          # your files live here: strategy.py, exports/, state/
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
RUNTIME = "Google Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local Jupyter"
print(f"repository root: {ROOT}\nworking folder:  {WORK}\nrunning in:      {RUNTIME}")

In [ ]:
# Imports, environment check and the exercise tracker
%matplotlib inline
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from quantsoc import market, features, modeling, portfolio, backtest, viz, widgets
from quantsoc.market import SYMBOLS, ASSET_NAMES, load_market, to_wide, session_series, describe_market
from quantsoc.features import FEATURE_NAMES, build_features, latest_features
from quantsoc.modeling import build_dataset, clean_rows, xy, split_boundaries, score_forecasts, predictions_wide, coefficient_table
from quantsoc.portfolio import PortfolioConfig, allocate_weights, check_weights, equal_weight
from quantsoc.backtest import evaluate_strategy, compare_table, decisions_from_forecasts, run_backtest, run_replay
from quantsoc.exercises import ExerciseTracker
from quantsoc.notebook_support import environment_report, setup_ipython, runtime_loss_notice, mode_banner, export_checkpoint, get_alpaca_credentials

setup_ipython()                      # short, readable message when a later cell needs an unfinished exercise
ex = ExerciseTracker()               # checks, hints and labelled reference solutions for the six exercises
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 20)
display(environment_report())
print("interactive widgets:", "available" if widgets.WIDGETS_AVAILABLE else "NOT available - static fallbacks will be used")
if IN_COLAB or IN_KAGGLE:
    runtime_loss_notice()

### A 30-second look at the finished system
Before building anything, here is what the end of the pipeline looks like: the portfolio value of a finished strategy
replayed at high speed over unseen data, with the allocation changing bar by bar. Everything that produces this
animation is code you will write or connect in the next two hours.

### Your fund
Give your fund a name and write down your **initial hypothesis** in one sentence: what pattern in prices do you
*guess* could predict what happens next? (We will test it, not trust it.)

🔮 **Predict:** if a stock rose over the last 15 minutes, do you expect it to keep rising, to fall back, or neither over the next 5 minutes?

In [ ]:
# Demo of the finished system (reference implementation, replayed over the final-test window)
bars = load_market(DATA / "workshop_market.csv")
_dataset = build_dataset(bars)
_model = modeling.fit_model(clean_rows(_dataset, "train"))
_forecasts = predictions_wide(_model, _dataset, "test")
_demo = evaluate_strategy(to_wide(bars, "open").loc[_forecasts.index], _forecasts, PortfolioConfig(threshold=0.0003))["net"]
HTML(viz.demo_animation(_demo.equity, _demo.weights, n_frames=24).to_jshtml())

In [ ]:
FUND_NAME = "Untitled Capital"                       # <- change me
HYPOTHESIS = "Assets that rose over the last few bars tend to keep rising for the next bar."   # <- your guess
FUND_SLUG = "".join(c.lower() if c.isalnum() else "_" for c in FUND_NAME).strip("_") or "my_fund"
print(f"Fund: {FUND_NAME}   (export folder name: {FUND_SLUG})\nHypothesis: {HYPOTHESIS}")

## 1. Understand the market

A **bar** summarises trading over a fixed interval. Our bars are **five minutes** long: each has an `open`, `high`,
`low`, `close` and `volume`. A bar labelled `09:30` covers 09:30–09:35; its close is only known at **09:35**.

A **return** is the fractional change in price: `return = price_now / price_before − 1`. A return of `0.001` is
0.1 %, also written as **10 basis points (bps)**; 1 bp = 0.01 %. We work in returns rather than prices because returns
of different assets are comparable and roughly stable over time, prices are not.

**Sessions.** Trading happens 09:30–16:00 New York time, 78 bars a day. There are no bars overnight; the first bar
of a day has no "previous bar" in its session. We will always respect that boundary.

**Asset explorer (below the table).** Normalised prices (everything starts at 100) make assets with different price
levels comparable. Notice how the five lines tend to move **together** at times: that is a shared *market factor*.

In [ ]:
print(f"{len(bars):,} rows = {bars['symbol'].nunique()} assets x {bars['timestamp'].nunique():,} bars "
      f"({bars['session'].nunique()} sessions of {market.BARS_PER_SESSION} five-minute bars)")
display(bars.head(6))
display(describe_market(bars).assign(name=pd.Series(ASSET_NAMES)))

closes   = to_wide(bars, "close")        # timestamp x symbol table of closes
opens    = to_wide(bars, "open")         # timestamp x symbol table of opens (the next tradable price)
sessions = session_series(bars)          # session label for every timestamp
ex.set_context(1, closes=closes, sessions=sessions); ex.set_context(2, closes=closes, sessions=sessions)
display(closes.tail(3))

# Asset explorer: normalised prices (start = 100) and return distributions. Use the controls to pick assets.
lib_returns = features.bar_returns(closes, sessions)      # library implementation; you write your own in Exercise 1
widgets.asset_explorer(closes, lib_returns)

### ✏️ Exercise 1: per-asset returns *(5 lines)*
Compute the return of every completed bar for every asset, **within each session**, as a `timestamp × symbol`
DataFrame. The first bar of every session must be `NaN` (there is no previous bar in that session).

Tip: `closes.groupby(sessions)` groups the rows of `closes` by session; pandas' `pct_change` computes
`price / previous price − 1`.

🔮 **Predict:** which asset will have the widest return distribution? (Look at the explorer above before you answer.)

In [ ]:
def compute_bar_returns(closes, sessions):
    """Return of each completed bar within its session: close[t] / close[t-1] - 1."""
    # YOUR CODE HERE (replace the next line)
    raise NotImplementedError("Exercise 1")


bar_returns = ex.attempt(1, compute_bar_returns, closes, sessions)

In [ ]:
# Checkpoint 1: return distributions (uses YOUR returns; if Exercise 1 is not complete this cell stops with a message)
bar_returns = ex.get(1)
print(bar_returns.describe().loc[["mean", "std"]].mul(1e4).round(2).rename(index={"mean": "mean (bps)", "std": "std (bps)"}))
print("NaN count per asset (should equal the number of sessions):", bar_returns.isna().sum().to_dict())
viz.return_histograms(bar_returns);

**Checkpoint 1 interpretation.** The five distributions are centred near zero (per-bar drift is tiny) but have very
different widths: ECHO moves about three times as much per bar as AURA. That *volatility* difference matters later
when we set position limits. Also note the fat tails: a few bars are much larger than a normal distribution would suggest.

## 2. Find a candidate signal

A **feature** is a number computed from *past* data that might carry information. A **target** is the *future*
quantity we want to predict. A **forecast** is the model's estimate of the target.

**Hypotheses vs accidental patterns.** With five assets and thousands of bars you can always find *some* pattern by
looking hard enough. That is why we (1) state the hypothesis first, (2) test it on data the model has never seen,
and (3) keep one slice of data locked away until the very end.

### The timing contract (read this twice)
1. Bar `t` is complete at its end; its close is the last thing we can observe.
2. Features for a decision at time `t` use closes of bars `≤ t` in the **same session**.
3. The first price we can actually trade at is the **open of bar `t+1`**, not the close we just observed.
4. Our order fills at `open[t+1]`; the next decision fills at `open[t+2]`.
5. So the return the strategy can *earn* from that decision is **`target[t] = open[t+2] / open[t+1] − 1`**.

Skipping step 3 is the most common backtesting mistake: it credits you with a price you could never have traded at.

### Worked example: a lagged feature
A feature must be *available* when the decision is made. `prev_return` (the return of the bar that just completed)
is computed by shifting *within the session*, so nothing leaks across days or from the future.

In [ ]:
# Worked example: build the research table = features + target + split label for every (bar, asset)
prev_return = lib_returns.groupby(sessions).shift(1)      # return of the bar BEFORE the one that just completed
print("prev_return at 09:40 equals the 09:35 bar's return:", np.isclose(prev_return.iloc[2, 0], lib_returns.iloc[1, 0]))

dataset = build_dataset(bars)                             # library: features (ret_1, mom_3, vol_12) + tradable target + splits
print("columns:", list(dataset.columns))
display(dataset[dataset.symbol == "AURA"].iloc[10:16])    # warm-up rows are NaN, then features appear
train_rows = clean_rows(dataset, "train")                 # only rows with complete features AND a target inside the split
print(f"training rows: {len(train_rows):,}  (after removing warm-up and boundary-crossing examples)")

### ✏️ Exercise 2: three-bar momentum *(3–5 lines)*
**Momentum** here means the return over the last three completed bars: `close[t] / close[t−3] − 1`, again within
each session. It is the feature behind the hypothesis "what went up keeps going up".

🔮 **Predict:** will momentum be *positively* or *negatively* related to the next tradable return in this market?

In [ ]:
def compute_momentum(closes, sessions, bars=3):
    """Return over the last `bars` completed bars within a session: close[t] / close[t-bars] - 1."""
    # YOUR CODE HERE (replace the next line)
    raise NotImplementedError("Exercise 2")


momentum_3 = ex.attempt(2, compute_momentum, closes, sessions)

### Can you spot the signal? *(training rows only)*
Left: every training example as a dot, which looks like noise. Right: the same data in equal-count buckets with 95 %
confidence intervals. A real signal shows up as a *trend in the bucket means*, not as visible structure in the cloud.
Switch between the three features. Only the **training** split is shown: we never look at validation or test data
while forming hypotheses.

Then **record your hypothesis** in the cell below: which feature would you bet on, and in which direction?

In [ ]:
SIGNAL_HYPOTHESIS = "Higher 3-bar momentum is followed by a higher next-bar return; the effect is small but consistent."   # <- edit
momentum_3 = ex.get(2)
lib = features.momentum(closes, sessions, 3)
print("your momentum equals the library feature 'mom_3' used from here on:", np.allclose(momentum_3.fillna(0), lib.fillna(0)))
display(widgets.signal_explorer(train_rows))

corrs = train_rows[FEATURE_NAMES + ["target"]].corr()["target"].drop("target")
print(SIGNAL_HYPOTHESIS)
print("\ncorrelation with the target on TRAINING rows:\n" + corrs.round(4).to_string())

## 3. Train a predictor

### Chronological splitting
We split **by time**, not at random: **train (first 60 %)** to fit, **validation (next 20 %)** to experiment and
choose settings, **final test (last 20 %)** which we look at exactly once, at the end. All five assets are split
at the same timestamps, and any example whose target window would cross a split boundary is removed.

Why not shuffle? Neighbouring bars are correlated, so a random split would let the model peek at the "future"
through its neighbours and produce a flattering, useless score.

### The zero-return baseline
Before fitting anything, ask: *how good is the forecast "nothing will happen"?* Its **mean squared error (MSE)** is
simply the average squared target. Any model we build must beat this number on **validation** data, otherwise it has
learned nothing useful.

In [ ]:
bounds = split_boundaries(dataset["timestamp"])
for name, (start, end) in bounds.items():
    n = clean_rows(dataset, name)
    print(f"{name:10s} {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M}   usable examples: {len(n):,}")
print("examples removed because their target crosses a split boundary:", int(dataset["crosses_split"].sum()))
viz.split_timeline(closes, bounds, "CRUX");

In [ ]:
# The zero-return baseline: how good is the forecast "nothing will happen"?
val_rows = clean_rows(dataset, "validation")
X_train, y_train = xy(train_rows)             # feature matrix in FEATURE_NAMES order, target vector
X_val, y_val = xy(val_rows)
baseline = score_forecasts(y_val, np.zeros_like(y_val))
print(f"zero-forecast baseline MSE on validation: {baseline.baseline_mse:.3e}  (that is a typical bar-return of about "
      f"{np.sqrt(baseline.baseline_mse) * 1e4:.0f} bps)")

### ✏️ Exercise 3: fit a linear model *(3–6 lines)*
Fit a **linear regression** that predicts the target from the three features. Use `X_train, y_train` **only**.
Optionally put a `StandardScaler` in front (fit on training data only; a `Pipeline` does this for you); it makes
the coefficients comparable across features.

🔮 **Predict:** by what percentage will the model reduce the baseline MSE on validation? 50 %? 5 %? 0.5 %?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def fit_linear_model(X_train, y_train):
    # YOUR CODE HERE: build the model, fit it on the TRAINING data only, return it
    raise NotImplementedError("Exercise 3")


ex.set_context(3, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val)
model = ex.attempt(3, fit_linear_model, X_train, y_train)

In [ ]:
# Checkpoint 3: validation forecasts vs the baseline
model = ex.get(3)
val_score = score_forecasts(y_val, model.predict(X_val))
display(val_score.as_frame())
try:
    display(coefficient_table(model)); viz.coefficient_bars(coefficient_table(model))
except (KeyError, AttributeError):
    print("(coefficient table needs the StandardScaler + LinearRegression pipeline; skipping)")
viz.forecast_vs_actual(y_val, model.predict(X_val));

**Reading checkpoint 3: uncertainty and errors.**
* The MSE improvement is a **few percent**, not fifty. The scatter is still a cloud. Yet the decile plot shows the
  highest forecasts *do* earn more on average than the lowest. In trading, an edge is usually this small: it only
  pays off through many, many decisions.
* The **hit rate** (sign correct) is a little above 50 %. A coin flip would give 50 %.
* Every individual forecast is wrong by roughly the full size of a typical return; the model knows the *average*
  behaviour, never the next bar.
* These are validation numbers. We have not looked at the final test, and we will not until the settings are frozen.

## 4. Build a portfolio

A forecast is not a trade. We need **weights**: the fraction of the portfolio's value held in each asset.
Whatever is not invested is **cash** (weight `1 − sum(weights)`). **Exposure** is the total invested fraction.

**Our rule (long-only, no leverage):**
1. Consider only assets whose forecast is finite and **above a threshold** (a tiny expected gain is not worth a trade).
2. Rank them by forecast; keep the best **up to 3** (`max_positions`). Ties are broken alphabetically so results are reproducible.
3. Give each selected asset `min(exposure_budget / 3, per_asset_cap)`.
4. Everything else stays in cash. We do **not** re-scale the survivors to use up the budget.

Example: budget 90 %, cap 30 %, two assets qualify → 30 % each, **40 % cash**.

The first cell shows the equal-weight benchmark and the reference rule on example forecasts. The second is
**"You are the portfolio manager"**: move the controls and watch how threshold, position limit, budget and cap change
the allocation **and the cash**. The "forecasts" dropdown switches between real validation bars, including one where
nothing qualifies and one where a forecast is missing (`NaN`).

In [ ]:
config = PortfolioConfig(threshold=0.0002, max_positions=3, exposure_budget=0.9, per_asset_cap=0.3, cost_bps=2.0)
ew = equal_weight(SYMBOLS, config)                       # the benchmark: spread the budget over all five assets
print("equal-weight benchmark:", ew.round(3).to_dict(), "| constraint report:", check_weights(ew, PortfolioConfig(max_positions=5)))
example = pd.Series({"AURA": 0.0008, "BOLT": 0.0001, "CRUX": 0.0004, "DUNE": -0.0003, "ECHO": np.nan})
print("reference allocation for example forecasts:", allocate_weights(example, config).to_dict())
viz.allocation_bar(allocate_weights(example, config), example, config, title="Reference rule on example forecasts");

In [ ]:
ex.require(3)
val_fc = predictions_wide(model, dataset, "validation")
_t = val_fc.dropna(how="all")
scenarios = {
    "a strong bar":        _t.loc[_t.max(axis=1).idxmax()],
    "a weak bar":          _t.loc[_t.max(axis=1).idxmin()],
    "a typical bar":       _t.iloc[len(_t) // 2],
    "a missing forecast":  _t.iloc[len(_t) // 3].mask(pd.Series([False, False, True, False, False], index=_t.columns)),
    "warm-up (all NaN)":   pd.Series(np.nan, index=_t.columns),
}
widgets.portfolio_manager(scenarios, allocate=allocate_weights, base=config)

### ✏️ Exercise 4: implement the allocation rule *(8–12 lines)*
Write `my_allocate(predictions, config)` returning a `pd.Series` of weights indexed by symbol (0 for unselected
assets, never `NaN`). `config` is a `PortfolioConfig` with `threshold`, `max_positions`, `exposure_budget`,
`per_asset_cap`. The checker runs your function on every edge case: all below threshold, fewer than three
qualifying, non-finite forecasts, binding caps and ties.

In [ ]:
def my_allocate(predictions, config):
    """Top `max_positions` forecasts above `threshold`, each min(budget / max_positions, cap); rest in cash."""
    preds = pd.Series(predictions, dtype=float)
    weights = pd.Series(0.0, index=preds.index)
    # YOUR CODE HERE:
    #  1. keep finite forecasts above config.threshold
    #  2. sort by forecast (highest first), break ties by symbol name, keep the top config.max_positions
    #  3. give each chosen symbol min(config.exposure_budget / config.max_positions, config.per_asset_cap)
    raise NotImplementedError("Exercise 4")
    return weights


my_allocate = ex.attempt(4, lambda: my_allocate)

In [ ]:
# Checkpoint 4: edge cases with YOUR function, then the manager controls driven by it
ex.require(3, 4)
my_allocate = ex.get(4)
cases = {
    "all below threshold": {"AURA": 0.0001, "BOLT": -0.001, "CRUX": 0.0, "DUNE": 0.00005, "ECHO": -0.0002},
    "two qualify":         {"AURA": 0.002, "BOLT": 0.0001, "CRUX": 0.001, "DUNE": -0.001, "ECHO": 0.0},
    "four qualify":        {"AURA": 0.002, "BOLT": 0.0015, "CRUX": 0.001, "DUNE": 0.0009, "ECHO": -0.001},
    "non-finite":          {"AURA": np.nan, "BOLT": 0.0015, "CRUX": np.inf, "DUNE": 0.0009, "ECHO": 0.0006},
    "cap binds (budget 100%, cap 25%)": {"AURA": 0.002, "BOLT": 0.0015, "CRUX": 0.001, "DUNE": 0.0009, "ECHO": 0.0008},
}
rows = []
for name, preds in cases.items():
    cfg = PortfolioConfig(exposure_budget=1.0, per_asset_cap=0.25) if "cap binds" in name else config
    w = my_allocate(pd.Series(preds), cfg)
    rep = check_weights(w, cfg)
    rows.append({"case": name, **{s: f"{w[s]:.0%}" for s in SYMBOLS}, "cash": f"{rep['cash']:.0%}", "constraints ok": rep["ok"]})
display(pd.DataFrame(rows).set_index("case"))
widgets.portfolio_manager(scenarios, allocate=my_allocate, base=config)

## 5. Evaluate

A **backtest** replays decisions over historical data with explicit rules for *when* you could trade and *what it
cost*. Ours (supplied) works like this:

* Decision after bar `t` completes → fill at `open[t+1]` (the timing contract).
* Holdings are shares; between fills their weights **drift** with prices, so each trade is computed against the
  *actual* current holdings, not against the previous target.
* **Cost** = `cost_bps / 10 000 × value traded`, per side, taken from cash at the fill. This is a simplification:
  real costs include spread, market impact and commissions, and depend on order size.
* **Turnover** = value traded per bar as a fraction of the portfolio. **Drawdown** = how far the value sits below its
  previous peak. We use equity, drawdown and turnover as the main metrics; annualised statistics over a few weeks of
  synthetic data would be misleading, so we do not compute them.
* During warm-up (no forecasts at all) the account **holds**: nothing is sold or bought.

The equal-weight benchmark is run through the **same** engine with the same timing and costs.

**Where did the profit go?** (second cell) The gap between the gross and net curves is what you pay to trade. Set the
cost level and press **Apply** (the decisions are cached; only the accounting re-runs).

🔮 **Predict:** at what cost per trade does the strategy stop making money on validation?

In [ ]:
ex.require(3, 4)
val_opens = opens.loc[val_fc.index]
val_results = evaluate_strategy(val_opens, val_fc, config, allocate=my_allocate, label=FUND_NAME)
display(compare_table(val_results))
viz.performance_panel(val_results, title=f"{FUND_NAME} on VALIDATION data");

In [ ]:
ex.require(3, 4)
val_decisions = decisions_from_forecasts(val_fc, config, allocate=my_allocate)     # cached target weights
widgets.cost_explorer(val_opens, val_decisions, config)

### ✏️ Exercise 5: choose one configuration and explain it *(a few lines + one sentence)*
Using **validation data only**, try two or three thresholds (and, if you like, the position limit or budget). Pick
one, and write one or two sentences of *evidence* for the choice. Then it is **frozen**: the same configuration runs
on the final test, on the stress scenario and in the trading application. No changes after peeking.

In [ ]:
ex.require(3, 4)

def try_config(threshold_bps, max_positions=3, exposure_budget=0.9, per_asset_cap=0.3, cost_bps=2.0):
    cfg = PortfolioConfig(threshold=threshold_bps / 1e4, max_positions=max_positions, exposure_budget=exposure_budget,
                          per_asset_cap=per_asset_cap, cost_bps=cost_bps)
    r = evaluate_strategy(val_opens, val_fc, cfg, allocate=my_allocate)["net"]
    return pd.Series({"net return %": 100 * r.total_return, "max drawdown %": 100 * r.max_drawdown,
                      "avg turnover %": 100 * r.avg_turnover, "costs $": r.total_costs}, name=f"{threshold_bps:g} bps")

display(pd.concat([try_config(t) for t in (1, 2, 3, 5)], axis=1).round(2))   # <- experiment here (validation only)

# YOUR CODE HERE: pick your configuration and justify it with validation evidence
my_config = PortfolioConfig(threshold=0.0002, max_positions=3, exposure_budget=0.9, per_asset_cap=0.3, cost_bps=2.0)
justification = ""     # one or two sentences of evidence from the table / cost explorer

frozen = ex.attempt(5, lambda: (my_config, justification))

In [ ]:
# FREEZE, then look at the final test exactly once
ex.require(3, 4, 5)
frozen_config, frozen_reason = ex.get(5)
print("FROZEN configuration:", frozen_config)
print("Reason:", frozen_reason)
test_fc = predictions_wide(model, dataset, "test")
test_opens = opens.loc[test_fc.index]
test_results = evaluate_strategy(test_opens, test_fc, frozen_config, allocate=my_allocate, label=FUND_NAME)
X_test, y_test = xy(clean_rows(dataset, "test"))
print("\nforecast quality on the FINAL TEST:")
display(score_forecasts(y_test, model.predict(X_test)).as_frame())
display(compare_table(test_results))
viz.performance_panel(test_results, title=f"{FUND_NAME} on the FINAL TEST (frozen settings)");

**Reading the final test.** Whatever the numbers are, they are one draw of one synthetic period: they tell you whether
the *process* held up out of sample, not what "the strategy earns". Compare with the equal-weight benchmark on the
same timing and costs: a market that simply rose during the test window rewards the benchmark too. Notice drawdown
and turnover, not only the final return.

## 6. Develop the application

Everything so far lives in **notebook state**: variables like `model`, `config`, `closes` exist only while this
kernel runs. A trading application cannot depend on that. It needs functions with **explicit inputs and outputs**:

```
predict_returns(history, model)  -> forecasts       # history = the completed bars the broker gave us
allocate(predictions, config)    -> target weights
```

The supplied engine imports exactly these two functions from a file called `strategy.py`, loads the fitted model
from `model.joblib` once (it does not retrain), and does the rest: reading positions, sizing orders, talking to the
broker, logging. Your job is to refactor your research into those two functions.

### Worked example: a reusable function
`latest_features(history)` (from the library, the same code that built the research table) returns one feature row
per symbol for the most recent completed bar. Notice it takes *only* `history` as input.

In [ ]:
ex.require(3)
history_example = bars[bars["timestamp"] <= closes.index[300]].tail(40 * 5)      # what a broker would hand us: the last 40 bars
latest = latest_features(history_example)
display(latest)
print("features are in the fixed order", FEATURE_NAMES, "-> model.predict(latest[FEATURE_NAMES]) =",
      model.predict(latest[FEATURE_NAMES].to_numpy(dtype=float)).round(6))

### ✏️ Exercise 6a: `predict_returns(history, model)` *(4–6 lines)*
The cell below **writes a file** (`%%writefile strategy.py`). That is the transparent way we turn notebook code into
a module; nothing is extracted behind your back. Complete `predict_returns`: symbols whose features are not ready
must stay `NaN`; every other symbol gets `model.predict(...)`. The file may only use its arguments and `quantsoc` imports.

In [ ]:
%%writefile strategy.py
"""My strategy: the two functions the trading engine calls (must not depend on notebook variables)."""
import numpy as np
import pandas as pd

from quantsoc.features import FEATURE_NAMES, latest_features
from quantsoc.portfolio import PortfolioConfig


def predict_returns(history: pd.DataFrame, model) -> pd.Series:
    latest = latest_features(history)                        # one feature row per symbol, FEATURE_NAMES order
    preds = pd.Series(np.nan, index=sorted(latest.index), dtype=float)
    # YOUR CODE HERE: rows whose features are all available -> model.predict -> write into preds
    raise NotImplementedError("Exercise 6a")
    return preds

### ✏️ Exercise 6b: `allocate(predictions, config)` *(copy your Exercise 4 code)*
This cell **appends** to `strategy.py`. Paste the body of your `my_allocate` function.
(If you re-run 6a the file is rewritten, so re-run 6b afterwards.)

In [ ]:
%%writefile -a strategy.py


def allocate(predictions: pd.Series, config: PortfolioConfig) -> pd.Series:
    preds = pd.Series(predictions, dtype=float)
    weights = pd.Series(0.0, index=preds.index)
    # YOUR CODE HERE (same rule as Exercise 4)
    raise NotImplementedError("Exercise 6b")
    return weights

In [ ]:
# Checkpoint 6: import strategy.py, check both functions, and PROVE it matches the research pipeline
ex.require(3, 4, 5)
from quantsoc.artifacts import load_strategy_module
_hist_points = [closes.index[i] for i in (5, 200, 700, 1400, 2300)]
_histories = [bars[bars["timestamp"] <= t].groupby("symbol").tail(40) for t in _hist_points]
for k in ("6a", "6b"):
    ex.set_context(k, strategy_path=WORK / "strategy.py", model=model, histories=_histories)
strategy = ex.attempt("6a", lambda: load_strategy_module(WORK / "strategy.py", name="strategy"))
strategy = ex.attempt("6b", lambda: load_strategy_module(WORK / "strategy.py", name="strategy"))
strategy = ex.get("6b")

# comparison checkpoint: exported functions vs research objects, on identical inputs from the validation window
rows = []
for t in val_fc.index[[20, 100, 250, 400, 550]]:
    hist = bars[bars["timestamp"] <= t].groupby("symbol").tail(40)
    p_new = strategy.predict_returns(hist, model)
    p_old = val_fc.loc[t]
    w_new = strategy.allocate(p_new, frozen_config)
    w_old = my_allocate(p_old, frozen_config)
    rows.append({"decision bar": f"{t:%m-%d %H:%M}", "max |forecast diff|": float((p_new - p_old).abs().max()),
                 "max |weight diff|": float((w_new - w_old).abs().max())})
parity = pd.DataFrame(rows).set_index("decision bar")
display(parity)
print("research and exported implementation are equivalent:", bool((parity.fillna(0) < 1e-12).all().all()))

### Export: code, model and configuration
The bundle contains **your actual `strategy.py`** (or the labelled reference copy if you used `ex.use_reference`),
the fitted model with its preprocessing, the feature names and timeframe, your frozen configuration, version metadata
and instructions for running locally, plus the engine and data so it runs on its own. It never contains credentials.

**Checkpoint: download now.** On Colab this triggers a download of the ZIP; on Kaggle the cell prints where to download
it from. The runtime and its files will not survive a disconnect.

**A complete mocked trading cycle** (second cell). Before touching any broker we run one full engine cycle against a
**mocked broker** with scripted, realistic responses: one order gets **rejected**, one is only **partially filled**.
Read the decision log: this is exactly what the runner prints in production, and it is why the engine re-plans from
*actual* positions every cycle.

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
from quantsoc.artifacts import export_bundle, check_zip
strategy_source = "reference" if any(ex.states[k].status == "reference" for k in ("6a", "6b")) else "student"
bundle_dir = export_bundle(WORK / "exports" / FUND_SLUG, WORK / "strategy.py", model, frozen_config, SYMBOLS,
                           strategy_source=strategy_source, exercise_status=ex.status_dict(), fund_name=FUND_NAME,
                           repo_root=ROOT, extra_notes=f"hypothesis: {SIGNAL_HYPOTHESIS}")
print(f"bundle written to {bundle_dir}   strategy source: {strategy_source.upper()}")
display(ex.summary())
zip_path = export_checkpoint(bundle_dir, WORK / f"{FUND_SLUG}_strategy", download=True)

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
from quantsoc.broker import MockBroker
from quantsoc.engine import TradingEngine
from quantsoc.state import StateStore
from quantsoc.artifacts import load_bundle

bundle = load_bundle(bundle_dir)                       # the engine loads model + config + strategy from the bundle, once
mode_banner("mock", "scripted broker responses; nothing leaves this notebook")
_t = closes.index[1500]
mock = MockBroker(cash=100_000, prices=opens.loc[closes.index[1501]].to_dict(), bars_df=bars,
                  now=(_t + pd.Timedelta(minutes=5)).to_pydatetime(), is_open=True)
mock.next_close = (_t + pd.Timedelta(hours=2)).to_pydatetime()
mock.reject_symbols = {"BOLT"}; mock.partial_fill = {"AURA": 0.5}
mock_engine = TradingEngine(mock, bundle.predict_returns, bundle.allocate, bundle.model,
                            PortfolioConfig(**{**bundle.config.to_dict(), "threshold": -1.0}),   # force trades for the demo
                            bundle.symbols, state=StateStore(WORK / "state_mock"), mode="replay", sleep=lambda s: None,
                            order_wait_seconds=0.1, poll_interval=0)
result = mock_engine.run_cycle()
print("\n".join(result.log))
print("\nstatus:", result.status, "|", result.reason, "\npositions now:", {p.symbol: round(p.qty, 2) for p in mock.positions()})

## 7. Connect and run

Three modes, always labelled truthfully:

| Mode | Data | Fills | Orders |
|---|---|---|---|
| **Offline replay** (default) | synthetic bars, revealed one at a time | simulated at the next open | none |
| **Paper preview** | real market data and your paper account | none | *proposed only* |
| **Paper execution** | real market data and your paper account | Alpaca paper matching engine | submitted to the **paper** endpoint after an explicit click |

**Replay** is deterministic and fast; it tests the *application*. **Paper trading** uses a real broker API with fake
money; it tests the *integration* (clocks, data feeds, order states) and is slow, noisy and market-hours only.
Neither says anything about real profitability.

### Accelerated replay with the inline monitor
The replay below reveals the final-test bars one at a time to **your** `strategy.py` (only the last 40 bars are
visible at each step, like a live feed) and executes through the same accounting engine as the backtest, so the
numbers cannot drift apart. Use play / pause / step / speed.

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
trace = run_replay(bars, strategy.predict_returns, strategy.allocate, model, frozen_config,
                   start=test_fc.index[0], end=test_fc.index[-1], lookback_bars=40)
_common = test_results["net"].equity.index.intersection(trace.equity.index)
print(f"replay steps: {len(trace.steps)}  |  replay equity == backtest equity on {len(_common)} shared bars:",
      bool(np.allclose(test_results["net"].equity.loc[_common], trace.equity.loc[_common])))
widgets.replay_monitor(trace)

### Optional: connect to Alpaca paper trading (supervised)
You need free Alpaca **paper** API keys. **Colab:** add them as secrets named `APCA_API_KEY_ID` and
`APCA_API_SECRET_KEY` in the 🔑 *Secrets* panel on the left and enable notebook access. **Kaggle:** *Add-ons → Secrets*,
same two names, attached to this notebook (the demo also needs Internet switched on). **Local:** put them in a
`.env` file (see `.env.example`). Never paste keys into a cell.

The five fictional assets are mapped to liquid real ETFs for the demonstration (`AURA→SPY, BOLT→QQQ, CRUX→IWM,
DUNE→DIA, ECHO→XLK`, configurable via `QSW_SYMBOL_MAP`). **This mapping proves nothing about predictive validity**:
the model was fitted on synthetic data. Order sizes use the broker's real prices and balances, never synthetic ones.
The free data plan provides the IEX feed, which is what the engine requests.

**Preview, then (optionally) submit: one bounded cycle.** **Preview** fetches completed bars, forecasts with your
strategy and shows the orders the engine *would* send, using the account's actual positions and buying power. Nothing
is submitted. **Submit** is a separate, explicit click that works exactly once per preview, only in paper mode, and is
disabled again immediately. "Run all" cannot press it. If the market is closed the engine says so and proposes nothing.
After a submission, the order status table shows what the paper broker did (filled, partially filled, rejected); run
**Preview** again to see the reconciled state.

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
from quantsoc.notebook_support import symbol_map_from_env
paper_engine = None
_key, _secret, _source = get_alpaca_credentials()
if _key:
    from quantsoc.broker import AlpacaPaperBroker, BrokerError
    try:
        paper_broker = AlpacaPaperBroker(_key, _secret, feed="iex")
        paper_engine = TradingEngine(paper_broker, bundle.predict_returns, bundle.allocate, bundle.model, bundle.config,
                                     bundle.symbols, symbol_map=symbol_map_from_env(), state=StateStore(WORK / "state"),
                                     mode="paper", strategy_label=FUND_NAME)
        mode_banner("paper-preview", f"credentials from {_source}; paper endpoint; nothing is submitted by this cell")
        acct, clock = paper_broker.account(), paper_broker.clock()
        print(f"paper account: equity ${acct.equity:,.2f}, cash ${acct.cash:,.2f}, buying power ${acct.buying_power:,.2f}")
        print(f"market open now: {clock.is_open}   next open: {clock.next_open}   next close: {clock.next_close}")
        for sym, msg in paper_engine.validate_instruments().items():
            print(f"  {sym} -> {msg}")
    except BrokerError as e:
        paper_engine = None
        mode_banner("offline", f"Alpaca connection failed ({e}); continuing offline")
else:
    mode_banner("offline", "no Alpaca credentials found - the paper demo is skipped; every result above is unaffected")

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
if paper_engine is not None:
    display(widgets.paper_panel(paper_engine))
else:
    print("Paper demo not available in this session (no credentials). Offline replay above covers the same engine.")

# Fallback without widgets: change the flag to True and run this cell BY HAND to submit the previewed orders once.
SUBMIT_FOR_REAL = False
if paper_engine is not None and SUBMIT_FOR_REAL:
    res = paper_engine.run_cycle(submit=True)
    print(res.status, res.reason); print("\n".join(res.log[-12:]))

### Your application, running on its own
**Cloud participants:** the engine below runs a few cycles of the offline replay through the *runner path* (broker
adapter → engine → state files) and the inline monitor reads those state files, the same files the local dashboard
reads. Your export ZIP contains everything to run this at home.

**Local participants:** open two terminals in your export folder (or the repository root):
```
python run_trader.py --mode replay --speed 20        # accelerated offline replay, Ctrl+C stops gracefully
streamlit run dashboard.py                           # http://localhost:8501 - reads state/, never trades
```

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
from quantsoc.broker import ReplayBroker
replay_broker = ReplayBroker(bars, start=test_fc.index[0])
runner = TradingEngine(replay_broker, bundle.predict_returns, bundle.allocate, bundle.model, bundle.config, bundle.symbols,
                       state=StateStore(WORK / "state"), mode="replay", sleep=lambda s: None, strategy_label=FUND_NAME)
mode_banner("replay", "runner path: ReplayBroker -> engine -> state files -> monitor")
for _ in range(60):
    runner.run_cycle()
    if not replay_broker.advance():
        break
widgets.state_monitor(WORK / "state")

## 8. Reveal and extend

### How the market was generated
For each asset *i* and bar *t* (inside a session), the close-to-close log return was

$$ r_{i,t} = \beta\,\mathrm{clip}(m_{i,t-1}) + \sigma_i\big(\rho\,F_t + \sqrt{1-\rho^2}\,\varepsilon_{i,t}\big) $$

* $m_{i,t-1}$: the sum of the previous three returns **in the same session** (the planted momentum), with $\beta = 0.07$
  and a bound so it can never explode;
* $F_t$: one shared **market factor** per bar (the co-movement you saw), loading $\rho = 0.55$;
* $\varepsilon_{i,t}$: asset-specific noise; $\sigma_i$ ranges from 9 to 28 bps per bar (AURA … ECHO);
* the open of each bar is the previous close times a small random gap, so the *next tradable price* is never the close you observed;
* no overnight bars: the first bar of a day carries a larger gap and momentum restarts.

So the hypothesis was true by construction, and the linear model recovered $\beta$ through `mom_3`. Real markets
have no guaranteed $\beta$; finding one that survives costs and time is the actual job.

### Your edge disappeared: the stress scenario
The stress data was generated by the **same code** with volatility doubled and the momentum effect **reversed**
($\beta = -0.05$). Your frozen strategy runs on it unchanged.

In [ ]:
ex.require(3, 4, 5, "6a", "6b")
stress_bars = load_market(DATA / "stress_market.csv")
stress_ds = build_dataset(stress_bars, fractions=(0.0, 0.0, 1.0))           # everything is "test": nothing is refitted
stress_fc = predictions_wide(model, stress_ds, "test")
stress_res = evaluate_strategy(to_wide(stress_bars, "open").loc[stress_fc.index], stress_fc, frozen_config, allocate=strategy.allocate, label=FUND_NAME)
Xs, ys = xy(clean_rows(stress_ds, "test"))
print("forecast quality on the STRESS scenario (frozen model):"); display(score_forecasts(ys, model.predict(Xs)).as_frame())
display(pd.DataFrame({"final test": test_results["net"].summary(), "stress": stress_res["net"].summary()}))
viz.stress_comparison(test_results["net"], stress_res["net"]);

**Discussion.** The forecasts now correlate *negatively* with what happens; the fund keeps trading on a pattern that
no longer exists and pays costs for the privilege. Nothing in the code warned us; only monitoring would. Real
regimes change without announcement, which is why live systems watch their own forecast quality and turnover.

### Where to go from here
* **Deployment:** `docs/vps_deployment.md` walks through running the runner on a small Linux server with a
  supervised process, secure credentials and a private dashboard.
* **Research:** add a feature (volume? the market factor itself?), try a different target horizon, add a stop rule.
  Always try it on validation first, and freeze before the test.
* **Risk:** position limits by volatility, a daily loss limit that flattens the book, a kill switch in the runner.
* **UI:** the Streamlit dashboard is ~150 lines; add a forecast-quality chart or an order history table.
* **Reading:** systematic trading is one corner of quant finance. Derivatives pricing, risk management and
  market microstructure are others; QuantSoc runs sessions on each.

Thank you for building your first fund. Keep the ZIP; it is a working application you own.